# 单热编码

在第一部分，我们从零起步，创建了一个预测冰激淋销量的**多层神经网络**；在第二部分，我们亲手搭建了一个**神经网络训练框架**，支持全连接网络，解决了数值回归任务；在第三部分，我们进一步扩展了框架，使其具备了处理分类任务、多维数据和提取局部特征（CNN）的能力。

到目前为止，我们一直在处理**静态数据**：每一组销量数据、每一张图片，都是彼此独立的个体，模型不需要关心它们的前后关系。

但还有另一类数据，叫做**序列数据**（Sequence Data）。序列数据的特点是：**顺序很重要，上下文很重要**。比如股价，我们不能只看今天的数字，还要参考过去几个月的走势；比如音乐，一个音符单独听没有意义，它的美感来自于前后音符的组合。

在所有序列数据中，最贴近人类智能、也最复杂的，莫过于**语言**。

从这一部分开始，我们将继续扩充神经网络训练框架，尝试处理文字信息：这也是通往大语言模型的第一步。

---

## 如何把文字变成数字？

神经网络只能处理数字，所以我们面临的第一个问题是：**怎么把文字转换成数字？**

### 方法一：索引编码

最直接的想法是：把所有单词排成一列，依次给它们编个号，这叫**索引编码**（Index Encoding）。

比如，词表里前三个词是“你”、“我”、“他”，编号就分别是 0、1、2。如果用 GB-2312 汉字库，一共 6763 个汉字，索引编码就从 0 编到 6762。

听起来不错，但有一个隐患：神经网络看到 0 和 1，会以为 1 比 0 大，也就是“我”比“你”更重要，但这显然不是我们想要的。词与词之间没有大小关系，索引编码无意中引入了一个不存在的排序。

### 方法二：单热编码

这时候，我们在上一部分（MNIST 手写数字识别）用过的**单热编码**（One-Hot Encoding）正好可以派上用场。

单热编码的思路很简单：把每个词映射成一个向量，向量的长度等于词表的大小。向量里只有一个位置是 1，其余全是 0，这个 1 的位置就对应这个词的索引。

以 GB-2312 汉字库为例，每个汉字变成一个长度 6763 的向量：

|  词汇  |  索引编码  |          单热编码向量          |
|:----:|:------:|:------------------------:|
|  你   |   0    | [1, 0, 0, 0, 0, ... , 0] |
|  我   |   1    | [0, 1, 0, 0, 0, ... , 0] |
|  他   |   2    | [0, 0, 1, 0, 0, ... , 0] |

这样，每个词都是一个独立的方向，互相之间没有大小、没有距离，解决了索引编码的问题。

``💡 单热编码简单直观，但有个明显的缺点：词表有多大，向量就有多长。英文常用词汇轻松超过 5 万，向量就得有 5 万维度。不仅内存消耗巨大，而且这些向量里绝大多数都是 0，极其稀疏、浪费资源。``

In [1]:
import csv
import re
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

这部分的代码与上一部分完全相同，直接沿用。

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

同样沿用上一部分的 `Dataset` 基类。文本数据的加载方式与图像不同，我们将在下面的 `IMDBDataset` 中定制 `load()` 方法。

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

## 数据

### IMDB 数据集

在这一部分，我们将使用来自 IMDB 的影评数据集。为了加快训练速度，我们同样准备了一个迷你版：**TinyIMDB**。

数据集里每条样本包含两个字段：

- **观众影评**（review）：一段文字，是观众写下的观后感；
- **观众态度**（sentiment）：一个标签，表示喜欢（positive，用 1 表示）或讨厌（negative，用 0 表示）。

我们的目标是：训练一个模型，读完一段影评之后，判断这位观众是喜欢还是讨厌这部电影。

#### 数据预处理

在加载数据时，我们会做三件事：

1. **数据清洗**：影评原文里混有 HTML 标签（比如 `<br />`）和标点符号，我们用正则表达式把它们清除掉；
2. **构建词表**：收集数据集里出现过的所有单词，按字母顺序排列，组成词表（vocabulary）。排序是为了保证每次运行时词表顺序一致；
3. **索引编码**：把每条影评里的每个单词替换成它在词表中的编号，得到词元序列（tokens）。

加载完成后，数据集对象会包含以下属性：

| 属性 | 说明 |
|------|------|
| `vocabulary` | 词表，所有单词的有序列表 |
| `word2index` | 正向索引表：单词 → 编号 |
| `index2word` | 反向索引表：编号 → 单词 |
| `tokens` | 词元表：每条影评转换后的编号序列 |

以及以下辅助函数：

| 函数 | 说明 |
|------|------|
| `encode()` | 将一段文字转换为索引编码列表 |
| `decode()` | 将索引编码列表转换回文字 |
| `onehot()` | 将一个单词的索引编码转换为单热编码向量 |
| `argmax()` | 将单热编码向量转换回索引编码 |

``💡 为什么不直接存单热编码？如果词表有 50,000 个词，每条影评有 100 个词，一条影评就需要 50,000 x 100 = 5,000,000 个数字来存储。数据集有几千条影评，内存开销会非常大。所以我们只存索引编码，在真正需要时再用 onehot() 实时转换。``

In [4]:
class IMDBDataset(Dataset):

    def __init__(self, filename):
        self.filename = filename
        super().__init__()

    def load(self):
        # 加载原始数据
        self.reviews = []
        self.sentiments = []
        with open(self.filename, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            # 跳过表头
            next(reader)
            for row in reader:
                # 数据清洗
                self.reviews.append(self.clean_text(row[0].lower()).split())
                self.sentiments.append([0 if row[1] == "negative" else 1])

        # 构建词表（排序以保证顺序一致）
        self.vocabulary = sorted(set(word for line in self.reviews for word in line))
        # 建立正向和反向索引表
        self.word2index = {word: index for index, word in enumerate(self.vocabulary)}
        self.index2word = {index: word for index, word in enumerate(self.vocabulary)}
        # 将影评转换为索引编码序列（词元表）
        self.tokens = [
            [self.word2index[word] for word in line if word in self.word2index]
            for line in self.reviews
        ]

    @staticmethod
    def clean_text(text):
        # 去除 HTML 标签
        text = re.sub(r'<[^>]+>', '', text)
        # 去除标点符号
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        return text

    def encode(self, text):
        words = self.clean_text(text.lower()).split()
        return [self.word2index[word] for word in words if word in self.word2index]

    def decode(self, tokens):
        return " ".join([self.index2word[index] for index in tokens])

    def onehot(self, index):
        vector = np.zeros(len(self.vocabulary))
        vector[index] = 1
        return vector

    @staticmethod
    def argmax(vector):
        return np.argmax(vector)

## 验证

### 加载数据集

加载 TinyIMDB，看看第一条影评长什么样：原始单词序列、转换后的索引编码、以及对应的情感标签。

In [5]:
dataset = IMDBDataset('tinyimdb.csv')

print(f'vocabulary：{len(dataset.vocabulary)}')
print(f'review：{dataset.reviews[0]}')
print(f'tokens：{dataset.tokens[0]}')
print(f'sentiment：{dataset.sentiments[0]}')

vocabulary：150
review：['this', 'movie', 'was', 'excellent', 'i', 'enjoyed', 'the', 'plot', 'and', 'acting', 'the', 'character', 'was', 'wonderful', 'recommend', 'screenplay', 'actor', 'actress', 'by', 'is', 'a']
tokens：[129, 89, 136, 50, 69, 48, 125, 104, 11, 2, 125, 27, 136, 145, 106, 111, 4, 5, 25, 75, 0]
sentiment：[1]


### 索引编码：文字 ↔ 数字

用 `encode()` 把一段文字转成索引编码列表，再用 `decode()` 转回来：

In [6]:
message = 'i recommend this film'
tokens = dataset.encode(message)
print(f'encode：{tokens}')
print(f'decode："{dataset.decode(tokens)}"')

encode：[69, 106, 129, 54]
decode："i recommend this film"


### 单热编码：索引 ↔ 向量

用 `onehot()` 把一个单词的索引转成单热向量，再用 `argmax()` 转回索引：

In [7]:
word = 'recommend'
index = dataset.word2index[word]
print(f'index：{index}')

vector = dataset.onehot(index)
print(f'one hot：{vector}')
print(f'argmax：{dataset.argmax(vector)}')

index：106
one hot：[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0.]
argmax：106


单词 'recommend' 的索引是 106，所以在长度为 150 的向量中，只有第 106 个位置是 1，其余全是 0。使用 `argmax()` 找到那个 1 的位置，就能还原出原来的索引。

---

## 课后练习

单热编码可以用于表示**输出标签**。在数值回归问题中，我们会用到单热编码吗？比如预测冰激淋销量的例子里，增加一个特征值来表示是否下雨，怎样用单热编码来表达？